# **1.Google Drive Setup**

# **Import Drive Module:**

This cell imports the Google Drive module, allowing the notebook to access and manage files stored in Drive.



In [ ]:
from google.colab import drive

# **Mount Google Drive:**

This command links Google Drive to Colab, enabling seamless access to Drive files.


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# **2.Environment Setup**
This cell installs `datasets` for loading Hugging Face datasets and `jiwer` for evaluating ASR models with WER, ensuring the environment is ready for data handling and performance assessment.


In [ ]:
!pip install datasets
!pip install jiwer
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from datasets import load_dataset
from jiwer import wer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.3 MB/s eta 0:00:00


# **3.Model Loading:**
This cell loads the pre-trained Wav2Vec2ForCTC model and its corresponding Wav2Vec2Processor from Hugging Face. These components are used for automatic speech recognition (ASR), enabling you to transcribe spoken language into text without the need to train a model from scratch.

In [ ]:
# Load pre-trained Wav2Vec 2.0 model and processor
model_name = "AndrewMcDowell/wav2vec2-xls-r-300m-arabic"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.03k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

# **4.Data Loading**

In [ ]:
import pandas as pd
#Define correct dataset path
test_audio_folder = "/content/drive/MyDrive/L2-KSU/dataset"

# Load CSV file containing test filenames and transcriptions
csv_path = "/content/drive/MyDrive/L2-KSU/dataset/l2-ksu-test.csv"
df = pd.read_csv(csv_path)

# **Audio Transcription Setup and Function Definition**
This cell prepares for audio transcription by installing and importing librosa, generating a dictionary that links each audio file to its ground truth text, and defining a function that leverages the Wav2Vec2 model to convert audio into text. The function loads and processes the audio, feeds it into the model to obtain predictions, decodes them into readable text, and returns a cleaned version of the transcription.

In [ ]:
!pip install librosa
import librosa
# Create ground truth dictionary
ground_truths = dict(zip(df["path"], df["text"]))

# Function to transcribe audio using wav2vec
def transcribe_audio(file_path):
    audio, _ = librosa.load(file_path, sr=16000)  # Load audio with 16kHz sample rate
    input_values = processor(audio, sampling_rate=16_000, return_tensors="pt").input_values  # Process audio
    with torch.no_grad():
        logits = model(input_values).logits  # Get model predictions
    predicted_ids = torch.argmax(logits, dim=-1)  # Get predicted IDs
    transcription = processor.batch_decode(predicted_ids)[0]  # Decode to text
    return transcription.lower().strip()  # Convert to lowercase for consistency

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

# **5.Evaluation**

These cells handle evaluation by importing the `wer` metric from the `jiwer` library and applying it to compare predicted transcriptions with reference texts. This offers a reliable metric to quantify and assess the model's transcription performance.


In [ ]:
# Compute WER
!pip install jiwer
import os
from jiwer import wer

wer_scores = []
for file_name, true_text in ground_truths.items():
    audio_path = os.path.join(test_audio_folder, file_name)

    if not os.path.exists(audio_path):
        print(f"Warning: {audio_path} not found!")
        continue  # Skip missing files

    print(f"Processing: {file_name}...")

    # Transcribe
    predicted_text = transcribe_audio(audio_path)
    # Compute WER for this sample
    # The line below is changed. Instead of using 'jiwer.wer', we use just 'wer'
    # since we have already imported the 'wer' function directly from the 'jiwer' module.
    score = wer(true_text, predicted_text)
    wer_scores.append(score)

    # Print intermediate results
    print(f"Ground Truth: {true_text}")
    print(f"Predicted: {predicted_text}")
    # The line below is changed to format the 'score' variable, not the 'wer' function
    print(f"WER: {score:.4f}\n")

# Calculate average WER
if wer_scores:
    baseline_wer = sum(wer_scores) / len(wer_scores)
    print(f"\nBaseline WER before fine-tuning: {baseline_wer:.4f}")
else:
    print("No valid test files found.")

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/males/speaker2/sentence9/wav/Sp2_M_S9_2.wav...
Ground Truth: صلى الله على نبينا محمد و على آله و صحبه أجمعين
Predicted: صل اللىعلى نبينا محبن وعالأ لوصحبي يجمعين
WER: 0.9091

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/females/speaker1/sentence10/wav/Sp1_F_S10_3.wav...
Ground Truth: سبحان الله و بحمده سبحان الله العظيم
Predicted: سبحان لو بحمد سبحان لالأعند
WER: 0.7143

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/females/speaker1/sentence3/wav/Sp1_F_S3_3.wav...
Ground Truth: صراط الذين أنعمت عليهم
Predicted: سلاة الدين أن ععمت أليهم
WER: 1.2500

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/males/speaker3/sentence1/wav/Sp3_M_S1_2.wav...
Ground Truth: الحمد لله رب العالمين الرحمن الرحيم
Predicted: لحمد له رب العالم اللرحمن الحيم
WER: 0.8333

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/females/speaker2/sentence3/wav/Sp